In [1]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

import pandas as pd

df = pd.read_csv("../data/raw/bd_grid_version_5.csv")
print(df.shape)   # expect (1850, 45)
df.head()


(1850, 45)


,Date,Day of the week,Year,Month,Max. Demand at eve. peak (Generation end),Max. Demand at eve. peak (Sub-station end),Highest Generation (Generation end),Minimum Generation (Generation end),Day-peak Generation (Generation end),Evening-peak Generation (Generation end),...,Rangpur_supply,Rangpur_load,Cumilla_demand,Cumilla_supply,Cumilla_load,Khulna_demand,Khulna_supply,Khulna_load,Holiday name,Holiday_cat
0,2019-11-21,Thursday,2019,11,8785,8110.0,8785.0,5647.0,7294.7,8785,...,208,0,701,701,0,953,953,0,No Holiday,0
1,2019-11-22,Friday,2019,11,7886,7344.0,7886.0,5283.5,5725.9,7886,...,201,0,694,694,0,908,908,0,No Holiday,0
2,2019-11-23,Saturday,2019,11,8827,8197.0,8827.0,5200.0,6731.7,8946,...,210,0,729,729,0,974,974,0,No Holiday,0
3,2019-11-24,Sunday,2019,11,8855,8224.0,8855.0,5470.0,7062.8,8855,...,216,0,722,722,0,983,983,0,No Holiday,0
4,2019-11-25,Monday,2019,11,8982,8356.0,8978.0,5500.0,7110.7,8982,...,219,0,735,735,0,981,981,0,No Holiday,0


In [2]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

full_range = pd.date_range(df["Date"].min(), df["Date"].max())
missing_dates = full_range.difference(df["Date"])
print(f"Expected days: {len(full_range)}, Actual rows: {len(df)}, Missing: {len(missing_dates)}")
print(missing_dates)


Expected days: 1867, Actual rows: 1850, Missing: 17
DatetimeIndex(['2020-01-02', '2020-03-27', '2020-03-28', '2023-01-23',
               '2023-03-15', '2023-04-01', '2023-05-08', '2023-05-16',
               '2023-07-05', '2024-06-03', '2024-07-01', '2024-07-18',
               '2024-07-19', '2024-07-20', '2024-07-21', '2024-07-22',
               '2024-07-23'],
              dtype='datetime64[us]', freq=None)


In [3]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

print(df.dtypes.value_counts())
df.dtypes.to_csv("../outputs/tables/dtypes_audit.csv")


int64             35
float64            7
str                2
datetime64[us]     1
Name: count, dtype: int64


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

null_counts = df.isna().sum()
print(null_counts[null_counts > 0])   # print empty


Series([], dtype: int64)


In [5]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

numeric_cols = [c for c in df.columns if any(k in c.lower() for k in ["load","generation","demand","supply"])]
print(df[numeric_cols].describe().T[["min","mean","max"]])
print("Negative values:", (df[numeric_cols] < 0).sum().sum())

const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns:", const_cols)


                                                min          mean      max
Max. Demand at eve. peak (Generation end)    6958.0  11670.265405  17200.0
Max. Demand at eve. peak (Sub-station end)   6752.0  11167.435870  16417.0
Highest Generation (Generation end)          6958.0  11580.174973  16477.0
Minimum Generation (Generation end)          4292.2   8363.687492  13298.0
Day-peak Generation (Generation end)         5504.8   9870.285373  14488.0
Evening-peak Generation (Generation end)     6958.0  11545.215135  16477.0
Minimum Generation Forecast up to 8:00 hrs.  2679.2   8359.954914  13298.0
Coal supply Limitation                          0.0    110.123243    444.0
Dhaka_demand                                 2079.0   3995.085946   5944.0
Dhaka_supply                                 2079.0   3969.446486   5869.0
Dhaka_load                                      0.0     25.956757   2002.0
Chattogram_demand                             611.0   1147.416216   1633.0
Chattogram_supply        

In [6]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

divisions = ["Dhaka", "Chattogram", "Rajshahi", "Mymensingh", "Sylhet",
             "Barishal", "Rangpur", "Cumilla", "Khulna"]

identity_check = pd.DataFrame({
    "division": divisions,
    "exact_match_pct": [
        ((df[f"{d}_demand"] - (df[f"{d}_supply"] + df[f"{d}_load"])).abs() < 1e-6).mean() * 100
        for d in divisions
    ]
})
print(identity_check)
identity_check.to_csv("../outputs/tables/accounting_identity_check.csv", index=False)


     division  exact_match_pct
0       Dhaka        99.729730
1  Chattogram        99.783784
2    Rajshahi        99.729730
3  Mymensingh        98.918919
4      Sylhet        99.081081
5    Barishal        99.837838
6     Rangpur        99.729730
7     Cumilla        99.405405
8      Khulna        99.783784


In [7]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

for d in divisions:
    df[f"{d}_shed"] = (df[f"{d}_load"] > 0).astype(int)


In [8]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

balance = pd.DataFrame({
    "division": divisions,
    "pct_shedding_days": [df[f"{d}_shed"].mean() * 100 for d in divisions],
    "max_mw": [df[f"{d}_load"].max() for d in divisions],
}).sort_values("pct_shedding_days", ascending=False)

print(balance)
balance.to_csv("../outputs/tables/class_balance.csv", index=False)


     division  pct_shedding_days  max_mw
3  Mymensingh          28.108108     665
0       Dhaka          19.081081    2002
7     Cumilla          17.243243     440
6     Rangpur          14.918919     272
4      Sylhet          11.891892     350
2    Rajshahi           9.459459     356
8      Khulna           8.648649     400
1  Chattogram           4.810811     500
5    Barishal           1.513514     110


In [9]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

train = df[df["Date"] <= "2023-12-31"].reset_index(drop=True)
test  = df[df["Date"] >= "2024-01-01"].reset_index(drop=True)

print(f"Train: {train.shape}, {train['Date'].min()} to {train['Date'].max()}")
print(f"Test:  {test.shape}, {test['Date'].min()} to {test['Date'].max()}")

train.to_csv("../data/raw/train_2019_2023.csv", index=False)
test.to_csv("../data/raw/test_2024.csv", index=False)


Train: (1493, 54), 2019-11-21 00:00:00 to 2023-12-31 00:00:00
Test:  (357, 54), 2024-01-01 00:00:00 to 2024-12-30 00:00:00
